<a href="https://colab.research.google.com/github/HLZHarry/Explainable_Alpha_Agent-Quality_Scorecard/blob/main/Phase_1_Initializing_the_Reasoning_Engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 1. Install
!pip install -q -U transformers accelerate bitsandbytes mcp langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 46.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 47.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 39.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [2]:
# 2. Mount Drive
# 2. Authenticate and Mount Drive
from google.colab import drive
import os

drive.mount('/content/drive')

# Check if we can see the drive
print(f"Drive mounted. Current directory: {os.getcwd()}")

Mounted at /content/drive
Drive mounted. Current directory: /content


In [6]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

In [7]:
model_id  = "meta-llama/Meta-Llama-3-8B-Instruct"

In [9]:
from huggingface_hub import login

# When you run this, it will ask for your token
login()

Token has not been saved to git credential helper.


In [11]:
# Configure 4-bit quatization for memory efficiency
bnb_config = BitsAndBytesConfig(
    load_in_4bit = True,
    bnb_4bit_use_double_quant = True,
    bnb_4bit_quant_type = "nf4",
    bnb_4bit_compute_dtype = torch.bfloat16
)

# Load Tokenizer and Model
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config = bnb_config,
    device_map="auto",
)

print("Brain successfully initialized!")

OSError: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct.
403 Client Error. (Request ID: Root=1-69b41499-6516969f69f9ce846afb7691;1159dd18-eda5-472e-ad93-b09587840dad)

Cannot access gated repo for url https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct/resolve/main/config.json.
Your request to access model meta-llama/Meta-Llama-3-8B-Instruct is awaiting a review from the repo authors.

## 1. `BitsAndBytesConfig`: The "Compressor" 📉

Standard AI models are massive. **Llama-3-8B**, for example, has 8 billion parameters. At 16-bit precision, this model requires ~**16GB of VRAM** just to load — often exceeding the limits of a free Colab GPU (typically 12–15GB).

`BitsAndBytesConfig` solves this with **4-bit Quantization:**

- **What it does:** Compresses 16-bit floats down to 4-bit integers
- **Why we use it:** Reduces memory footprint by nearly **75%**, allowing the model to fit into ~**5.5GB of VRAM** — leaving plenty of headroom for data and long-term alpha calculations

---

## 2. Why This Model? (`Llama-3-8B-Instruct`) 🧠

Three specific reasons drove this choice:

| # | Reason | Why It Matters |
|---|---|---|
| **1** | **Reasoning vs. Size** | Widely considered one of the "smartest" models for its size — punches well above its weight class in logic and instruction-following |
| **2** | **Instruction Following** | The `-Instruct` variant is fine-tuned to act as an assistant, enabling the specific **Chain-of-Thought** process our agent requires |
| **3** | **Ecosystem Support** | Its popularity means tools like `LangChain` and the `MCP SDK` have out-of-the-box support for its specific quirks |